# 03 · Load Reference Data

**What this does:** loads the newest copies of two title reference files from NBC into BigQuery:

| File | What it holds | Where you save it |
|---|---|---|
| **Enrichment title table** | Titles, IMDb IDs, distributor, genre, widest release, PVOD and EST dates | `gs://nbcu-marcus-hilliard/enrichments_table_stage/<YYYYMMDD>_enrichments_title.csv` |
| **Standard CHTT (Finance)** | Finance's window dates and premium revenue per title | `gs://nbcu-marcus-hilliard/std_chtt_table_stage/he_ent_std_chtt_<YYYYMMDD>.csv` |

| | |
|---|---|
| **When to run** | After you copy a new version of either file from NBC into the folder above |
| **How long** | Under 1 minute |
| **What it changes** | Only the raw tables `nbc_enrichment_title` and `finance_std_chtt`, and only when `DRY_RUN = False` |
| **Safe to re-run?** | Yes. A file that is already loaded is skipped. |

### How to use it
1. Leave the settings in **Step 1** as they are and click **Run ▸ Run All Cells**. This is a *check only*.
2. Read the message in **Step 2**. If every file says **✅ ready**, set `DRY_RUN = False` and **Run All** again.
3. Look at **Step 3** to confirm the copies are in the tables.

> These files only *feed* the title review. Approved dates never change until someone approves them in the review step.

## Step 1 · Settings

In [1]:
# Which file to load: "all" (recommended), "enrichment_title" or "std_chtt"
SOURCE = "all"

# True  = check only, nothing is changed (always start here)
# False = actually load the data
DRY_RUN = False

# Advanced: leave these as False unless told otherwise
FORCE_RELOAD = False            # True = reload a file even if it was already loaded
ALLOW_SMALLER_FILE = False      # True = load even if much smaller than the previous copy

## Step 2 · Run the checks and the load
*Nothing to edit here.*

In [2]:
from IPython.display import Markdown, display

from us_hment_mmm.reference.load_csv import run, show_reference_summary

try:
    results = run(SOURCE, dry_run=DRY_RUN, force=FORCE_RELOAD, allow_smaller=ALLOW_SMALLER_FILE)
except Exception as exc:
    display(Markdown(f"### 🛑 STOP — the load failed\n\n`{exc}`\n\nThe existing tables are "
                     "unchanged. Save this notebook and send it to the pipeline owner."))
    raise

icons = {"READY": "✅ ready to load", "SUCCESS": "✅ loaded", "WOULD_SKIP": "ℹ️ already loaded",
         "SKIPPED": "ℹ️ already loaded", "WOULD_BLOCK": "🛑 cannot be loaded", "BLOCKED": "🛑 not loaded"}
lines = []
for r in results:
    line = f"- **{r['source']}** ({r['file'] or 'no file'}): {icons.get(r['status'], r['status'])}"
    if r.get("rows_loaded"):
        line += f", {r['rows_loaded']:,} rows"
    for p in r["problems"]:
        line += f"\n    - {p}"
    lines.append(line)
statuses = {r["status"] for r in results}
if statuses & {"WOULD_BLOCK", "BLOCKED"}:
    head = "### 🛑 STOP — at least one file has a problem"
    tail = ("**What to do:** a much smaller file usually means the copy from NBC was cut short; "
            "copy it again. For other problems, contact the pipeline owner.")
elif "READY" in statuses:
    head, tail = "### ✅ PASS — ready to load", "**Next:** in Step 1 set `DRY_RUN = False`, then **Run All** again."
else:
    head, tail = "### ✅ PASS", "See Step 3."
display(Markdown(head + "\n" + "\n".join(lines) + "\n\n" + tail))

REFERENCE LOAD  (LOAD)

--- enrichment_title  (NBC enrichment title table (titles, distributor, genre, window dates))
  found 20260830_enrichments_title.csv                   1.81 MB
  found 20261019_enrichments_title.csv                   3.76 MB
  found 20261030_enrichments_title.csv                   3.77 MB
  selected 20261030_enrichments_title.csv  (9 columns)
  loaded 86,908 rows (counts match) -> nbc_enrichment_title

--- std_chtt  (NBC Finance Standard CHTT (Finance window dates and premium revenue))
  found he_ent_std_chtt_20261030.csv                     0.24 MB
  selected he_ent_std_chtt_20261030.csv  (23 columns)
  loaded 1,282 rows (counts match) -> finance_std_chtt


### ✅ PASS
- **enrichment_title** (20261030_enrichments_title.csv): ✅ loaded, 86,908 rows
- **std_chtt** (he_ent_std_chtt_20261030.csv): ✅ loaded, 1,282 rows

See Step 3.

## Step 3 · Check the result
Copies held in each reference table (newest first).

In [3]:
display(show_reference_summary())

Source,Copy date,File,Rows
enrichment_title,2026-10-30,20261030_enrichments_title.csv,"86,908"
std_chtt,2026-10-30,he_ent_std_chtt_20261030.csv,"1,282"


---
### Running this as a Domino Job instead
```
python -m us_hment_mmm.reference.load_csv
```
Add `--dry-run` to check only.